In [2]:
import pandas as pd 
import requests
from jsonschema import validate
import json
import fastparquet

In [4]:
pd.set_option('display.width', None)

In [3]:
countriesDf = pd.DataFrame(pd.read_parquet('../dataRawBeforeBlob/countries/euCountriesRaw.parquet'))
countriesDf

,Code,Title,ParentDimension,Dimension,ParentCode,ParentTitle
0,AUT,Austria,REGION,COUNTRY,EUR,Europe
1,BEL,Belgium,REGION,COUNTRY,EUR,Europe
2,BGR,Bulgaria,REGION,COUNTRY,EUR,Europe
3,CYP,Cyprus,REGION,COUNTRY,EUR,Europe
4,CZE,Czechia,REGION,COUNTRY,EUR,Europe
5,DEU,Germany,REGION,COUNTRY,EUR,Europe
6,DNK,Denmark,REGION,COUNTRY,EUR,Europe
7,ESP,Spain,REGION,COUNTRY,EUR,Europe
8,EST,Estonia,REGION,COUNTRY,EUR,Europe
9,FIN,Finland,REGION,COUNTRY,EUR,Europe


In [5]:
baseURL= 'https://ghoapi.azureedge.net/api/'

In [9]:
lifeExpectancyIndicatorsDf = pd.DataFrame(pd.read_parquet('../dataRawBeforeBlob/indicators/lifeexpectancyIndicators.parquet',engine='fastparquet'))
lifeExpectancyIndicatorsDf

,IndicatorCode,IndicatorName,Language,Category
0,WHOSIS_000004,Adult mortality rate (probability of dying bet...,EN,Mortality and global health estimates
1,WHOSIS_000002,Healthy life expectancy (HALE) at birth (years),EN,Mortality and global health estimates
2,WHOSIS_000001,Life expectancy at birth (years),EN,Mortality and global health estimates
3,MDG_0000000007,Under-five mortality rate (probability of dyin...,EN,Child mortality and causes of death
4,WHOSIS_000003,Neonatal mortality rate (per 1000 live births),EN,Child mortality and causes of death
5,MORTADO,Adolescent mortality rate (per 1 000 age speci...,EN,Child mortality and causes of death


In [10]:
def getData(baseurl,indicatorCode:str):
    indicatorCode = indicatorCode.strip()
    response = requests.get(baseurl+f"{indicatorCode}")
    if response.status_code == 200 and response.headers.get('Content-Type').startswith('application/json') :
        return response.json()
    else:
        return 'Something went wrong with request'

In [11]:
def saveToParquet(df: pd.DataFrame,filename:str):
    df.to_parquet(f'../dataRawBeforeBlob/lifeExpectancyData/{filename}.parquet',engine='fastparquet')

In [12]:
firstIndicator = lifeExpectancyIndicatorsDf.iloc[0]
firstIndicator

IndicatorCode                                        WHOSIS_000004
IndicatorName    Adult mortality rate (probability of dying bet...
Language                                                        EN
Category                     Mortality and global health estimates
Name: 0, dtype: object

In [18]:
firstIndicatorData = getData(baseURL,firstIndicator['IndicatorCode'])
firstIndicatorData = firstIndicatorData['value']
firstIndicatorData

[{'Id': 3406,
  'IndicatorCode': 'WHOSIS_000004',
  'SpatialDimType': 'GLOBAL',
  'SpatialDim': 'GLOBAL',
  'ParentLocationCode': None,
  'TimeDimType': 'YEAR',
  'ParentLocation': None,
  'Dim1Type': 'SEX',
  'Dim1': 'SEX_BTSX',
  'TimeDim': 2004,
  'Dim2Type': None,
  'Dim2': None,
  'Dim3Type': None,
  'Dim3': None,
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '176',
  'NumericValue': 175.9135015,
  'Low': None,
  'High': None,
  'Comments': None,
  'Date': '2024-10-31T10:23:48.47+01:00',
  'TimeDimensionValue': '2004',
  'TimeDimensionBegin': '2004-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2004-12-31T00:00:00+01:00'},
 {'Id': 3492,
  'IndicatorCode': 'WHOSIS_000004',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'WSM',
  'ParentLocationCode': 'WPR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Western Pacific',
  'Dim1Type': 'SEX',
  'Dim1': 'SEX_FMLE',
  'TimeDim': 2014,
  'Dim2Type': None,
  'Dim2': None,
  'Dim3Type': None,
  'Dim3': None,
  'DataSource

In [19]:
firstIndicatorDataDataFrame = pd.DataFrame(firstIndicatorData)
firstIndicatorDataDataFrame = firstIndicatorDataDataFrame.loc[firstIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
firstIndicatorDataDataFrame = firstIndicatorDataDataFrame.loc[firstIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
firstIndicatorDataDataFrame = firstIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
firstIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
8,14916,WHOSIS_000004,EST,2001,SEX_BTSX,224
9,16254,WHOSIS_000004,SWE,2020,SEX_BTSX,49
16,22326,WHOSIS_000004,ESP,2007,SEX_BTSX,74
36,42203,WHOSIS_000004,BGR,2010,SEX_BTSX,143
66,86423,WHOSIS_000004,CYP,2018,SEX_BTSX,52
...,...,...,...,...,...,...
12886,10122648,WHOSIS_000004,ROU,2003,SEX_BTSX,174
12889,10128530,WHOSIS_000004,DNK,2004,SEX_BTSX,95
12897,10136020,WHOSIS_000004,ROU,2015,SEX_BTSX,136
12900,10141754,WHOSIS_000004,IRL,2012,SEX_BTSX,67


In [20]:
saveToParquet(firstIndicatorDataDataFrame,'adultmortalityratebetween15and60per1000')

In [21]:
secondIndicator = lifeExpectancyIndicatorsDf.iloc[1]
secondIndicator

IndicatorCode                                      WHOSIS_000002
IndicatorName    Healthy life expectancy (HALE) at birth (years)
Language                                                      EN
Category                   Mortality and global health estimates
Name: 1, dtype: object

In [22]:
secondIndicatorData = getData(baseURL,secondIndicator['IndicatorCode'])
secondIndicatorData = secondIndicatorData['value']
secondIndicatorData

[{'Id': 9454832,
  'IndicatorCode': 'WHOSIS_000002',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'NZL',
  'TimeDimType': 'YEAR',
  'ParentLocationCode': 'WPR',
  'ParentLocation': 'Western Pacific',
  'Dim1Type': 'SEX',
  'TimeDim': 2019,
  'Dim1': 'SEX_MLE',
  'Dim2Type': None,
  'Dim2': None,
  'Dim3Type': None,
  'Dim3': None,
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '69.5 [68.8-70.2]',
  'NumericValue': 69.49638181,
  'Low': 68.77259172,
  'High': 70.24810922,
  'Comments': None,
  'Date': '2024-08-02T10:11:00.407+02:00',
  'TimeDimensionValue': '2019',
  'TimeDimensionBegin': '2019-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2019-12-31T00:00:00+01:00'},
 {'Id': 281,
  'IndicatorCode': 'WHOSIS_000002',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'SVK',
  'TimeDimType': 'YEAR',
  'ParentLocationCode': 'EUR',
  'ParentLocation': 'Europe',
  'Dim1Type': 'SEX',
  'TimeDim': 2000,
  'Dim1': 'SEX_MLE',
  'Dim2Type': None,
  'Dim2': None,
  'Dim3Type': None

In [23]:
secondIndicatorDataDataFrame = pd.DataFrame(secondIndicatorData)
secondIndicatorDataDataFrame = secondIndicatorDataDataFrame.loc[secondIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
secondIndicatorDataDataFrame = secondIndicatorDataDataFrame.loc[secondIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
secondIndicatorDataDataFrame['Value'] = secondIndicatorDataDataFrame['Value'].astype('string')
secondIndicatorDataDataFrame['Value'] = secondIndicatorDataDataFrame['Value'].str.split(' ').str[0]
secondIndicatorDataDataFrame['Value'] = secondIndicatorDataDataFrame['Value'].astype('float')
secondIndicatorDataDataFrame = secondIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
secondIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
20,16156,WHOSIS_000002,EST,2017,SEX_BTSX,67.9
47,32499,WHOSIS_000002,BGR,2010,SEX_BTSX,64.5
52,37159,WHOSIS_000002,EST,2005,SEX_BTSX,63.5
56,38726,WHOSIS_000002,LTU,2014,SEX_BTSX,64.7
77,61677,WHOSIS_000002,LUX,2015,SEX_BTSX,71.1
...,...,...,...,...,...,...
12852,10135601,WHOSIS_000002,NLD,2021,SEX_BTSX,70.0
12862,10142828,WHOSIS_000002,POL,2016,SEX_BTSX,67.5
12893,10171679,WHOSIS_000002,LTU,2007,SEX_BTSX,61.6
12896,10174113,WHOSIS_000002,SVK,2006,SEX_BTSX,64.8


In [24]:
saveToParquet(secondIndicatorDataDataFrame,'haleAtBirth')

In [25]:
thirdIndicator = lifeExpectancyIndicatorsDf.iloc[2]
thirdIndicator

IndicatorCode                            WHOSIS_000001
IndicatorName         Life expectancy at birth (years)
Language                                            EN
Category         Mortality and global health estimates
Name: 2, dtype: object

In [26]:
thirdIndicatorData = getData(baseURL,thirdIndicator['IndicatorCode'])
thirdIndicatorData = thirdIndicatorData['value']
thirdIndicatorData

[{'Id': 871,
  'IndicatorCode': 'WHOSIS_000001',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'SOM',
  'TimeDimType': 'YEAR',
  'ParentLocationCode': 'EMR',
  'ParentLocation': 'Eastern Mediterranean',
  'Dim1Type': 'SEX',
  'Dim1': 'SEX_MLE',
  'TimeDim': 2008,
  'Dim2Type': None,
  'Dim2': None,
  'Dim3Type': None,
  'Dim3': None,
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '48.0 [46.7-49.6]',
  'NumericValue': 48.03754082,
  'Low': 46.71678387,
  'High': 49.62845721,
  'Comments': None,
  'Date': '2024-08-02T09:43:39.193+02:00',
  'TimeDimensionValue': '2008',
  'TimeDimensionBegin': '2008-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2008-12-31T00:00:00+01:00'},
 {'Id': 1065,
  'IndicatorCode': 'WHOSIS_000001',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'BTN',
  'TimeDimType': 'YEAR',
  'ParentLocationCode': 'SEAR',
  'ParentLocation': 'South-East Asia',
  'Dim1Type': 'SEX',
  'Dim1': 'SEX_BTSX',
  'TimeDim': 2002,
  'Dim2Type': None,
  'Dim2': None,
  'D

In [27]:
thirdIndicatorDataDataFrame = pd.DataFrame(thirdIndicatorData)
thirdIndicatorDataDataFrame = thirdIndicatorDataDataFrame.loc[thirdIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
thirdIndicatorDataDataFrame = thirdIndicatorDataDataFrame.loc[thirdIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
thirdIndicatorDataDataFrame['Value'] = thirdIndicatorDataDataFrame['Value'].astype('string')
thirdIndicatorDataDataFrame['Value'] = thirdIndicatorDataDataFrame['Value'].str.split(' ').str[0]
thirdIndicatorDataDataFrame['Value'] = thirdIndicatorDataDataFrame['Value'].astype('float')
thirdIndicatorDataDataFrame = thirdIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
thirdIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
71,59333,WHOSIS_000001,CYP,2010,SEX_BTSX,81.2
73,60532,WHOSIS_000001,MLT,2017,SEX_BTSX,82.1
83,71458,WHOSIS_000001,DNK,2013,SEX_BTSX,80.2
90,80280,WHOSIS_000001,HRV,2003,SEX_BTSX,74.9
91,80468,WHOSIS_000001,LTU,2003,SEX_BTSX,71.8
...,...,...,...,...,...,...
12831,10131572,WHOSIS_000001,HUN,2000,SEX_BTSX,71.4
12839,10137983,WHOSIS_000001,HUN,2009,SEX_BTSX,74.2
12856,10152241,WHOSIS_000001,IRL,2009,SEX_BTSX,79.5
12922,10192327,WHOSIS_000001,MLT,2010,SEX_BTSX,80.5


In [28]:
saveToParquet(thirdIndicatorDataDataFrame,'lifeexpetancyatbirth')

In [29]:
fourthIndicator = lifeExpectancyIndicatorsDf.iloc[3]
fourthIndicator

IndicatorCode                                       MDG_0000000007
IndicatorName    Under-five mortality rate (probability of dyin...
Language                                                        EN
Category                       Child mortality and causes of death
Name: 3, dtype: object

In [30]:
fourthIndicatorData = getData(baseURL,fourthIndicator['IndicatorCode'])
fourthIndicatorData = fourthIndicatorData['value']
fourthIndicatorData

[{'Id': 9454885,
  'IndicatorCode': 'MDG_0000000007',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'DMA',
  'ParentLocationCode': 'AMR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Americas',
  'Dim1Type': 'SEX',
  'TimeDim': 2022,
  'Dim1': 'SEX_FMLE',
  'Dim2Type': 'AGEGROUP',
  'Dim2': 'AGEGROUP_YEARSUNDER5',
  'Dim3Type': 'WEALTHQUINTILE',
  'Dim3': 'WEALTHQUINTILE_TOTL',
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '32.8 [28.4-38.0]',
  'NumericValue': 32.81284074,
  'Low': 28.401476804,
  'High': 38.035805741,
  'Comments': None,
  'Date': '2025-04-15T12:59:44.083+02:00',
  'TimeDimensionValue': '2022',
  'TimeDimensionBegin': '2022-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2022-12-31T00:00:00+01:00'},
 {'Id': 9455009,
  'IndicatorCode': 'MDG_0000000007',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'TON',
  'ParentLocationCode': 'WPR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Western Pacific',
  'Dim1Type': 'SEX',
  'TimeDim': 1995,
  'Dim1': '

In [35]:
fourthIndicatorDataDataFrame = pd.DataFrame(fourthIndicatorData)
fourthIndicatorDataDataFrame = fourthIndicatorDataDataFrame.loc[fourthIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
fourthIndicatorDataDataFrame = fourthIndicatorDataDataFrame.loc[fourthIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
fourthIndicatorDataDataFrame['Value'] = fourthIndicatorDataDataFrame['Value'].astype('string')
fourthIndicatorDataDataFrame['Value'] = fourthIndicatorDataDataFrame['Value'].str.split(' ').str[0]
fourthIndicatorDataDataFrame['Value'] = fourthIndicatorDataDataFrame['Value'].astype('float')
fourthIndicatorDataDataFrame = fourthIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
fourthIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
65,11306,MDG_0000000007,ESP,1997,SEX_BTSX,6.1
91,16349,MDG_0000000007,BEL,1954,SEX_BTSX,48.1
144,23113,MDG_0000000007,FRA,1990,SEX_BTSX,9.0
166,26559,MDG_0000000007,CYP,1988,SEX_BTSX,11.3
224,36329,MDG_0000000007,HUN,1998,SEX_BTSX,11.1
...,...,...,...,...,...,...
62972,10187750,MDG_0000000007,SVK,1970,SEX_BTSX,29.5
62976,10188500,MDG_0000000007,AUT,1968,SEX_BTSX,29.8
62978,10188599,MDG_0000000007,POL,1982,SEX_BTSX,22.6
62979,10188794,MDG_0000000007,AUT,1962,SEX_BTSX,37.9


In [38]:
saveToParquet(fourthIndicatorDataDataFrame,'underfivemortalityrate')

In [37]:
fifthIndicator = lifeExpectancyIndicatorsDf.iloc[4]
fifthIndicator

IndicatorCode                                     WHOSIS_000003
IndicatorName    Neonatal mortality rate (per 1000 live births)
Language                                                     EN
Category                    Child mortality and causes of death
Name: 4, dtype: object

In [39]:
fifthIndicatorData = getData(baseURL,fifthIndicator['IndicatorCode'])
fifthIndicatorData = fifthIndicatorData['value']
fifthIndicatorData

[{'Id': 9455167,
  'IndicatorCode': 'WHOSIS_000003',
  'SpatialDimType': 'UNICEFREGION',
  'SpatialDim': 'UNICEFREGIONEAP',
  'TimeDimType': 'YEAR',
  'ParentLocationCode': None,
  'ParentLocation': None,
  'Dim1Type': 'SEX',
  'TimeDim': 2005,
  'Dim1': 'SEX_BTSX',
  'Dim2Type': 'AGEGROUP',
  'Dim2': 'AGEGROUP_DAYS0-27',
  'Dim3Type': None,
  'Dim3': None,
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '14.9 [14.2-15.7]',
  'NumericValue': 14.899013971,
  'Low': 14.187388317,
  'High': 15.667195177,
  'Comments': None,
  'Date': '2025-04-15T16:15:21.28+02:00',
  'TimeDimensionValue': '2005',
  'TimeDimensionBegin': '2005-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2005-12-31T00:00:00+01:00'},
 {'Id': 639,
  'IndicatorCode': 'WHOSIS_000003',
  'SpatialDimType': 'WORLDBANKINCOMEGROUP',
  'SpatialDim': 'WB_HI',
  'TimeDimType': 'YEAR',
  'ParentLocationCode': None,
  'ParentLocation': None,
  'Dim1Type': 'SEX',
  'TimeDim': 2007,
  'Dim1': 'SEX_BTSX',
  'Dim2Type': 'A

In [40]:
fifthIndicatorDataDataFrame = pd.DataFrame(fifthIndicatorData)
fifthIndicatorDataDataFrame = fifthIndicatorDataDataFrame.loc[fifthIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
fifthIndicatorDataDataFrame = fifthIndicatorDataDataFrame.loc[fifthIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
fifthIndicatorDataDataFrame['Value'] = fifthIndicatorDataDataFrame['Value'].astype('string')
fifthIndicatorDataDataFrame['Value'] = fifthIndicatorDataDataFrame['Value'].str.split(' ').str[0]
fifthIndicatorDataDataFrame['Value'] = fifthIndicatorDataDataFrame['Value'].astype('float')
fifthIndicatorDataDataFrame = fifthIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
fifthIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
29,29247,WHOSIS_000003,AUT,1991,SEX_BTSX,4.4
32,31529,WHOSIS_000003,HRV,2020,SEX_BTSX,2.9
41,36081,WHOSIS_000003,POL,1970,SEX_BTSX,18.3
48,42565,WHOSIS_000003,POL,1981,SEX_BTSX,13.8
54,49925,WHOSIS_000003,LTU,1999,SEX_BTSX,4.8
...,...,...,...,...,...,...
11783,10133827,WHOSIS_000003,FIN,1982,SEX_BTSX,4.5
11795,10145198,WHOSIS_000003,ROU,2022,SEX_BTSX,3.2
11832,10174716,WHOSIS_000003,MLT,2009,SEX_BTSX,4.6
11864,10196434,WHOSIS_000003,ITA,1960,SEX_BTSX,24.7


In [41]:
saveToParquet(fifthIndicatorDataDataFrame,'neonatalMortalityRate')

In [42]:
sixthIndicator = lifeExpectancyIndicatorsDf.iloc[5]
sixthIndicator

IndicatorCode                                              MORTADO
IndicatorName    Adolescent mortality rate (per 1 000 age speci...
Language                                                        EN
Category                       Child mortality and causes of death
Name: 5, dtype: object

In [44]:
sixthIndicatorData = getData(baseURL,sixthIndicator['IndicatorCode'])
sixthIndicatorData = sixthIndicatorData['value']
sixthIndicatorData

[{'Id': 9454801,
  'IndicatorCode': 'MORTADO',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'SVK',
  'ParentLocationCode': 'EUR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Europe',
  'Dim1Type': 'SEX',
  'Dim1': 'SEX_MLE',
  'TimeDim': 2022,
  'Dim2Type': 'AGEGROUP',
  'Dim2': 'AGEGROUP_YEARS10-14',
  'Dim3Type': None,
  'Dim3': None,
  'DataSourceDimType': None,
  'DataSourceDim': None,
  'Value': '0.7 [0.6-0.9]',
  'NumericValue': 0.720128232,
  'Low': 0.589192267,
  'High': 0.86507728,
  'Comments': None,
  'Date': '2025-04-14T17:51:01.93+02:00',
  'TimeDimensionValue': '2022',
  'TimeDimensionBegin': '2022-01-01T00:00:00+01:00',
  'TimeDimensionEnd': '2022-12-31T00:00:00+01:00'},
 {'Id': 9455166,
  'IndicatorCode': 'MORTADO',
  'SpatialDimType': 'COUNTRY',
  'SpatialDim': 'KAZ',
  'ParentLocationCode': 'EUR',
  'TimeDimType': 'YEAR',
  'ParentLocation': 'Europe',
  'Dim1Type': 'SEX',
  'Dim1': 'SEX_MLE',
  'TimeDim': 1996,
  'Dim2Type': 'AGEGROUP',
  'Dim2': 'AGEGROUP_YEARS15

In [46]:
sixthIndicatorDataDataFrame = pd.DataFrame(sixthIndicatorData)
sixthIndicatorDataDataFrame = sixthIndicatorDataDataFrame.loc[sixthIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
sixthIndicatorDataDataFrame = sixthIndicatorDataDataFrame.loc[(sixthIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX') & (sixthIndicatorDataDataFrame['Dim2'] == 'AGEGROUP_YEARS10-19')]
sixthIndicatorDataDataFrame['Value'] = sixthIndicatorDataDataFrame['Value'].astype('string')
sixthIndicatorDataDataFrame['Value'] = sixthIndicatorDataDataFrame['Value'].str.split(' ').str[0]
sixthIndicatorDataDataFrame['Value'] = sixthIndicatorDataDataFrame['Value'].astype('float')
sixthIndicatorDataDataFrame = sixthIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
sixthIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
110,13955,MORTADO,LTU,2016,SEX_BTSX,3.4
155,20067,MORTADO,CYP,1994,SEX_BTSX,4.2
212,27113,MORTADO,LTU,1995,SEX_BTSX,6.7
268,34177,MORTADO,EST,1991,SEX_BTSX,7.9
346,43925,MORTADO,LUX,2001,SEX_BTSX,2.1
...,...,...,...,...,...,...
75937,10173322,MORTADO,HRV,2021,SEX_BTSX,2.3
75959,10176508,MORTADO,DNK,2009,SEX_BTSX,1.8
76024,10184348,MORTADO,IRL,2004,SEX_BTSX,3.2
76092,10192613,MORTADO,BEL,2003,SEX_BTSX,2.8


In [ ]:
saveToParquet(sixthIndicatorDataDataFrame,'adolescentmortalityrate10-19')